In [0]:
# Set default catalog and schema so we can use short table names
spark.sql("USE CATALOG nyc_taxi_project")
spark.sql("USE SCHEMA raw")

DataFrame[]

In [0]:
# Path to the landing volume where you uploaded the monthly parquet files
raw_path = "/Volumes/nyc_taxi_project/raw/taxi_landing/"

# Read ALL files matching this pattern — this is what makes it "dynamic":
# whenever you drop a new month's file in here, this same line picks it up
df = spark.read.parquet(raw_path + "yellow_tripdata_*.parquet")

# Quick sanity check — see the schema and a few rows
df.printSchema()
display(df.limit(10))

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
1,2024-03-01T00:18:51.000,2024-03-01T00:23:45.000,0,1.3,1,N,142,239,1,8.6,3.5,0.5,2.7,0.0,1.0,16.3,2.5,0.0
1,2024-03-01T00:26:00.000,2024-03-01T00:29:06.000,0,1.1,1,N,238,24,1,7.2,3.5,0.5,3.0,0.0,1.0,15.2,2.5,0.0
2,2024-03-01T00:09:22.000,2024-03-01T00:15:24.000,1,0.86,1,N,263,75,2,7.9,1.0,0.5,0.0,0.0,1.0,10.4,0.0,0.0
2,2024-03-01T00:33:45.000,2024-03-01T00:39:34.000,1,0.82,1,N,164,162,1,7.9,1.0,0.5,1.29,0.0,1.0,14.19,2.5,0.0
1,2024-03-01T00:05:43.000,2024-03-01T00:26:22.000,0,4.9,1,N,263,7,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0
2,2024-03-01T00:50:42.000,2024-03-01T01:10:40.000,1,5.04,1,N,238,159,2,25.4,1.0,0.5,0.0,0.0,1.0,27.9,0.0,0.0
2,2024-03-01T00:08:23.000,2024-03-01T00:17:53.000,1,2.15,1,N,161,141,1,12.1,1.0,0.5,5.13,0.0,1.0,22.23,2.5,0.0
2,2024-03-01T00:24:58.000,2024-03-01T00:30:31.000,1,1.1,1,N,236,237,1,8.6,1.0,0.5,2.04,0.0,1.0,15.64,2.5,0.0
2,2024-03-01T00:49:40.000,2024-03-01T01:01:25.000,1,2.78,1,N,161,114,1,14.9,1.0,0.5,2.0,0.0,1.0,21.9,2.5,0.0
1,2024-03-01T00:21:43.000,2024-03-01T00:24:44.000,1,0.3,1,N,237,141,2,5.1,3.5,0.5,0.0,0.0,1.0,10.1,2.5,0.0


In [0]:
from pyspark.sql.functions import current_timestamp, col

# Add audit columns:
# - ingestion_time: when this row was loaded
# - source_file: which physical file this row came from (from Unity Catalog's built-in _metadata column)
df = df.withColumn("ingestion_time", current_timestamp()) \
       .withColumn("source_file", col("_metadata.file_path"))

In [0]:
table_name = "nyc_taxi_project.raw.bronze_taxi_trips"

try:
    if spark.catalog.tableExists(table_name):
        df.createOrReplaceTempView("new_data")
        spark.sql(f"""
            MERGE INTO {table_name} AS t
            USING new_data AS s
            ON t.source_file = s.source_file
               AND t.VendorID = s.VendorID
               AND t.tpep_pickup_datetime = s.tpep_pickup_datetime
            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"✅ Merge succeeded at {spark.sql('SELECT current_timestamp()').collect()[0][0]}")
    else:
        df.write.format("delta").saveAsTable(table_name)
        print("✅ Table created for the first time.")
except Exception as e:
    print(f"❌ Bronze pipeline failed: {str(e)}")
    raise

✅ Merge succeeded at 2026-07-09 03:22:42.972957


In [0]:
result = spark.table("nyc_taxi_project.raw.bronze_taxi_trips")
print(f"Total rows in bronze table: {result.count()}")
result.groupBy("source_file").count().show(truncate=False)

Total rows in bronze table: 9554778
+-------------------------------------------------------------------------------+-------+
|source_file                                                                    |count  |
+-------------------------------------------------------------------------------+-------+
|dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-02.parquet|3007526|
|dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-01.parquet|2964624|
|dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet|3582628|
+-------------------------------------------------------------------------------+-------+

